# Hartree-Fock (HF) in the atomic limit

In [180]:
using MatsubaraFunctions
using NLsolve

T = 0.2
U = 0.2
N = 2000

mf = MatsubaraMesh(T, N, Fermion);
G  = MeshFunction(mf; data_t=ComplexF64);

Helper function: Matsubara sum of a MeshFunction ``f(ν)``. 
Shall only have one mesh, namely for the fermionic Matsubara frequencies.

In [187]:
function matsubara_sum(f::MeshFunction)
    mf = f.meshes[1]
    T = temperature(mf)
    s  = zero(eltype(f.data))
    for i in eachindex(mf)
        s += f[i]                 # f[i] == f(points(mf, i))
    end
    return T*s + 0.5
end

matsubara_sum (generic function with 1 method)

Initialize $G_0(ν) = 1/(iν)$:

In [192]:
for i in eachindex(mf)
    ω = value(value(points(mf, i)))
    G[i] = 1.0 / (im*ω)
end

Fixed-point: $n = T \sum_{iν} G(iν) e^{iν0+}$; and $G(ν) = \frac{1}{iν - U n}$


In [ ]:
function fixed_point!(F, nvec, G)
    n = nvec[1]
    for i in eachindex(mf)
        ω = value(value(points(mf, i)))
        G[i] = 1.0 / (im*ω - U*n)
    end
    F[1] = real(matsubara_sum(G)) - n     
end

fixed_point! (generic function with 2 methods)

In [191]:
n0 = [1.0]
res = nlsolve((F, n) -> fixed_point!(F, n, G), n0, method=:anderson)
n_HF = res.zero[1]

0.4010663288259362